In [25]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    db_path='faq.db'
)

In [26]:
len(files)

72

In [28]:
question = "How does the agentic loop keep calling the model until it stops?"


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [31]:
from minsearch import Index

index=Index(text_fields=['content'],keyword_fields=['filename']).fit(documents)


In [32]:
index.search(question)[0] #QUESTION 2

{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry 

In [ ]:
from rag_helper import RAGBase
class RAGDoc(RAGBase):
    def search(self, query, num_results=5):
        boost_dict = {'filename': 3.0, 'content': 0.5}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
        )
    def build_context(self, search_results):
        lines=[]
        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

ragdoc = RAGDoc(index=index, llm_client=openai_client)
question = "How does the agentic loop keep calling the model until it stops?"
ragdoc.rag(question)  # warm up cache
response_unchunked = ragdoc.rag(question)

In [ ]:
print(response_unchunked["output_text"])

In [35]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks) #Question 4

295

In [ ]:
index.fit(chunks)
ragdoc = RAGDoc(index=index, llm_client=openai_client)
question = "How does the agentic loop keep calling the model until it stops?"
ragdoc.rag(question)  # warm up cache
response_chunked = ragdoc.rag(question)
print(response_chunked['input_tokens_details'])

In [ ]:
unchunked_cached_tokens = response_unchunked['input_tokens_details'].cached_tokens
chunked_cached_tokens = response_chunked['input_tokens_details'].cached_tokens
unchunked_cached_tokens / chunked_cached_tokens

In [38]:
def search(query):
    """search docs for a fitting source of info"""
    boost_dict = {'filename': 3.0, 'content': 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [39]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the course documentation for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course docs.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}
from toyaikit.tools import Tools
agent_tools = Tools()
agent_tools.add_tool(search)

instructions='''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
'''

In [40]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

-> Response received


-> Response received
